<a href="https://colab.research.google.com/github/anniewanjohi/SemProject_MLEngine/blob/main/LogisticRegression_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AI powered Internship and placement matching mobile app
171001-Annie Wanjohi
190384-Sankok Seyianoi
Base Model-logical Regression

In [1]:
!pip install datasets pandas -q

from datasets import load_dataset
import pandas as pd

dataset = load_dataset("cnamuangtoun/resume-job-description-fit")
df = dataset["train"].to_pandas()
df.head()

train.csv: reconstructing file:   0%|          |  0.00B / 53.4MB            

train.csv: downloading bytes:           |  0.00B            

test.csv: reconstructing file:   0%|          |  0.00B / 15.2MB            

test.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6241 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1759 [00:00<?, ? examples/s]

,resume_text,job_description_text,label
0,SummaryHighly motivated Sales Associate with e...,Net2Source Inc. is an award-winning total work...,No Fit
1,Professional SummaryCurrently working with Cat...,At Salas OBrien we tell our clients that were ...,No Fit
2,SummaryI started my construction career in Jun...,Schweitzer Engineering Laboratories (SEL) Infr...,No Fit
3,SummaryCertified Electrical Foremanwith thirte...,"Mizick Miller & Company, Inc. is looking for a...",No Fit
4,SummaryWith extensive experience in business/r...,Life at Capgemini\nCapgemini supports all aspe...,No Fit


Data Loading and inspection


In [2]:
# 2. Size and structure
print(df.shape)
df.info()

(6241, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6241 entries, 0 to 6240
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   resume_text           6241 non-null   object
 1   job_description_text  6241 non-null   object
 2   label                 6241 non-null   object
dtypes: object(3)
memory usage: 146.4+ KB


In [3]:
# 3. Missing values and empty text
print(df.isnull().sum())
print((df["resume_text"].str.strip() == "").sum())
print((df["job_description_text"].str.strip() == "").sum())

resume_text             0
job_description_text    0
label                   0
dtype: int64
0
0


In [ ]:
# 4. Duplicates
print("Duplicate rows:", df.duplicated().sum())

In [4]:
# 5. Class balance
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True))

label
No Fit           3143
Potential Fit    1556
Good Fit         1542
Name: count, dtype: int64
label
No Fit           0.503605
Potential Fit    0.249319
Good Fit         0.247076
Name: proportion, dtype: float64


In [5]:
# 6. Unique resumes and job descriptions
print("Unique resumes:", df["resume_text"].nunique())
print("Unique jobs:", df["job_description_text"].nunique())

Unique resumes: 642
Unique jobs: 280


Exploratory Data Analysis(EDA)

In [6]:
# 7. Text length
df["resume_len"] = df["resume_text"].str.len()
df["job_len"] = df["job_description_text"].str.len()
df[["resume_len", "job_len"]].describe()

,resume_len,job_len
count,6241.000000,6241.000000
mean,5804.402980,2731.328313
std,2989.766702,1716.078138
min,897.000000,72.000000
25%,4234.000000,1231.000000
50%,5134.000000,2384.000000
75%,6608.000000,3941.000000
max,25364.000000,7669.000000


Resumes are about twice as long as job descriptions (mean 5,804 vs 2,731 characters), and resume length ranges from 897 to 25,364. This matters because when the two texts are combined, the resume’s words could outweigh the job’s. The shortest job description is only 72 characters, so I will check whether it is a real description before modelling.

In [7]:
# 8. Read full texts (not truncated)
pd.set_option("display.max_colwidth", None)
df["resume_text"][0]

"SummaryHighly motivated Sales Associate with extensive customer service and sales experience. Outgoing sales professional with track record of driving increased sales, improving buying experience and elevating company profile with target market.\nHighlights-Soft Skills: Public Speaking, Public Relations, Team Building, Project Management, Procedure writing, Staff Supervision and Management, Ability to interface with professionals on all levels. Accomplishments, Honors, and Activities -Board of Directors Member for the Food Bank of Corpus Christi from November 2010 to April 2013. -Held Life Insurance License -Basketball Official (Referee) High School Varsity Level.\nExperienceAccountant,08/2014-05/2015Aspirus–Owen,WI,Perform daily and routine accounting functions for two main companies and five small royalty companies. Responsibilities include but are not limited to the following: Accounts Payable, Accounts Receivable, Manage and reconcile funds for multiple banks accounts, Payroll, Pe

Reading the first resume in full shows words stuck together (“SummaryHighly”, “ExperienceAccountant,08/2014...”), literal \n line breaks, section headings like “Highlights” and “Experience”, and dates mixed into the text. A model that counts words would treat these as real words, so I will fix them in the cleaning stage.

In [8]:
# 9. Are the rows sorted by label?
df["label"].head(20).tolist()

['No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit',
 'No Fit']

The first 20 labels are all “No Fit”, which suggests the rows are ordered by label. I will shuffle and stratify the data before splitting, otherwise the training and test sets could contain different classes.

In [11]:
df_clean = df.copy()
print("Starting shape:", df_clean.shape)

Starting shape: (6241, 5)


In [13]:
#Step 1 Duplicates
print("Duplicate rows:", df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates()
print("Shape after dropping duplicates:", df_clean.shape)

Duplicate rows: 1
Shape after dropping duplicates: (6240, 5)


we checked for exact duplicate rows and removed them. The dataset went from 6,241 to 6,240 rows, so only one duplicate was found

In [14]:
#Step 2 Scrubbing for irrelevant data
df_clean = df_clean.drop(columns=["resume_len", "job_len"], errors="ignore")
print(df_clean.columns.tolist())

['resume_text', 'job_description_text', 'label']


All three original columns (resume text, job description text, label) are needed for the model, so none were dropped. The two length columns were only for exploring, and we exclude them when modelling.

In [15]:
#Step 3 Scrub for incorrect data
df_clean["job_len"] = df_clean["job_description_text"].str.len()
df_clean["resume_len"] = df_clean["resume_text"].str.len()

# Look at the shortest job descriptions
print(df_clean.sort_values("job_len").head(5)[["job_description_text", "job_len", "label"]])


                                                          job_description_text  \
824   I own a small landscaping business that needs help with accountingtaxes.   
4426  I own a small landscaping business that needs help with accountingtaxes.   
3315  I own a small landscaping business that needs help with accountingtaxes.   
3414  I own a small landscaping business that needs help with accountingtaxes.   
4496  I own a small landscaping business that needs help with accountingtaxes.   

      job_len          label  
824        72         No Fit  
4426       72  Potential Fit  
3315       72  Potential Fit  
3414       72  Potential Fit  
4496       72  Potential Fit  


We checked the shortest job descriptions because one was only 72 characters. It turned out to be a real but very short posting from a small business owner, so we kept it

In [19]:
import re

def fix_text(text):
    text = text.replace("\\n", " ")                    # literal \n characters
    text = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", text)   # split glued words like SummaryHighly
    text = re.sub(r"\s+", " ", text).strip()           # remove extra spaces
    return text

df_clean["resume_text"] = df_clean["resume_text"].apply(fix_text)
df_clean["job_description_text"] = df_clean["job_description_text"].apply(fix_text)

The text had glued words (such as “SummaryHighly”), literal \n characters and extra spaces from how it was collected. We inserted spaces between joined words and cleaned the whitespace. Some joined words, like “accountingtaxes”, cannot be fixed by this rule, and the rule can wrongly split names like “JavaScript”

Before and after

In [20]:
print(df["resume_text"][0][:300])
print("---")
print(df_clean["resume_text"].iloc[0][:300])

SummaryHighly motivated Sales Associate with extensive customer service and sales experience. Outgoing sales professional with track record of driving increased sales, improving buying experience and elevating company profile with target market.
Highlights-Soft Skills: Public Speaking, Public Relati
---
Summary Highly motivated Sales Associate with extensive customer service and sales experience. Outgoing sales professional with track record of driving increased sales, improving buying experience and elevating company profile with target market. Highlights-Soft Skills: Public Speaking, Public Relat


In [21]:
#Handle missing data
print(df_clean.isnull().sum())
df_clean = df_clean.dropna()

resume_text             0
job_description_text    0
label                   0
job_len                 0
resume_len              0
dtype: int64


no missing data.So no rows were dropped or filled

In [22]:
#Step 6: Treat outliers
Q1 = df_clean["resume_len"].quantile(0.25)
Q3 = df_clean["resume_len"].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR
lower = Q1 - 1.5 * IQR

outliers = df_clean[(df_clean["resume_len"] > upper) | (df_clean["resume_len"] < lower)]
print("Upper limit:", upper, "| Number of outliers:", len(outliers))

Upper limit: 10169.0 | Number of outliers: 510


We used the IQR rule on resume length, which gave an upper limit of 10,169 characters. 510 resumes were above it. We kept them, because they look like genuinely detailed resumes and not errors, and removing them would have cut about 8% of the data.

In [23]:
#Step 7: Standardise and normalise
df_clean["resume_text"] = df_clean["resume_text"].str.lower()
df_clean["job_description_text"] = df_clean["job_description_text"].str.lower()

label_map = {"No Fit": 0, "Potential Fit": 1, "Good Fit": 2}
df_clean["label_encoded"] = df_clean["label"].map(label_map)
print(df_clean["label_encoded"].value_counts())

label_encoded
0    3142
1    1556
2    1542
Name: count, dtype: int64


We converted all text to lowercase so that “Python” and “python” count as the same word. We also encoded the labels as numbers (No Fit = 0, Potential Fit = 1, Good Fit = 2). Scaling is not needed for the text columns, since the conversion to numbers later handles that.

In [24]:
print("Final shape:", df_clean.shape)
df_clean.head()

df_clean.to_csv("clean_data.csv", index=False)

Final shape: (6240, 6)


In [26]:
#Joining text into one input
df_model = df_clean[["resume_text", "job_description_text", "label_encoded"]].reset_index(drop=True)
df_model["text"] = df_model["resume_text"] + " " + df_model["job_description_text"]
X = df_model["text"]
y = df_model["label_encoded"]

In [27]:
from sklearn.model_selection import GroupShuffleSplit

groups = df_model["resume_text"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

print(len(X_train), len(X_test))
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

4917 1323
label_encoded
0    0.504983
1    0.247915
2    0.247102
Name: proportion, dtype: float64
label_encoded
0    0.498110
1    0.254724
2    0.247166
Name: proportion, dtype: float64


In [28]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", max_features=20000)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=20000, stop_words='english')),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [29]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(model, X_train, y_train, groups=groups_train,
                        cv=cv, scoring=["accuracy", "f1_macro"])

print("Accuracy per fold:", scores["test_accuracy"])
print("Mean accuracy:", scores["test_accuracy"].mean())
print("Macro F1 per fold:", scores["test_f1_macro"])
print("Mean macro F1:", scores["test_f1_macro"].mean())

Accuracy per fold: [0.53922453 0.53452381 0.51652502 0.60731707 0.59504132]
Mean accuracy: 0.5585263510432616
Macro F1 per fold: [0.53358275 0.51079888 0.50644739 0.6057879  0.59127007]
Mean macro F1: 0.5495773975144389


In [30]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Baseline accuracy (always guess No Fit):", dummy.score(X_test, y_test))

y_pred = model.predict(X_test)
print("Model accuracy:", model.score(X_test, y_test))
print(classification_report(y_test, y_pred,
      target_names=["No Fit", "Potential Fit", "Good Fit"]))
print(confusion_matrix(y_test, y_pred))

Baseline accuracy (always guess No Fit): 0.4981103552532124
Model accuracy: 0.5699168556311414
               precision    recall  f1-score   support

       No Fit       0.62      0.57      0.60       659
Potential Fit       0.47      0.47      0.47       337
     Good Fit       0.58      0.67      0.62       327

     accuracy                           0.57      1323
    macro avg       0.56      0.57      0.56      1323
 weighted avg       0.57      0.57      0.57      1323

[[378 161 120]
 [143 157  37]
 [ 90  18 219]]


In [31]:
print("Train accuracy:", model.score(X_train, y_train))
print("Test accuracy:", model.score(X_test, y_test))

Train accuracy: 0.6802928615009152
Test accuracy: 0.5699168556311414


To check that our Logistic Regression results can be trusted, we will train two other models, Naive Bayes and a Support Vector Machine, on the same data.

Each model will use the same cleaned data, the same split and the same cross-validation, so the comparison is fair.

We will put the scores in one table, next to the baseline that always guesses No Fit (about 50%). If the other models score about the same as Logistic Regression, that supports our results. If one scores much higher, we can improve our model.